# PDHub Mutagenesis (BIOS6380)

We work through one clinical variant from sequence to call: BRCA1 p.A1708E, a missense in the BRCT1 phospho-peptide pocket. The point of the exercise is that structure prediction alone is silent on this kind of variant, and that an actual pathogenicity call has to combine structure, sequence, and conservation evidence.

What you will do:
1. Predict the WT and mutant structures with the ESMFold API.
2. Read pLDDT, AlphaMissense, and conservation at the variant residue.
3. Combine the evidence using the ACMG/AMP framework (Richards 2015) and look at where each tool fails.

Runs in about two minutes on a free Colab CPU. No GPU, no local models. The structure prediction is a remote API call; everything else is lookup and arithmetic.

The notebook is laid out as six sections: setup, predict, score, conservation, verdict, critique. The conservation, AlphaMissense, and Findlay 2018 functional data are cached for the BRCA1 A1708E worked example; everything else adapts if you change the variant in section 0.

Background reading: Buel and Walters (2022) Nat Struct Mol Biol 29:1–2, "Can AlphaFold2 predict the impact of missense mutations on structure?". This notebook is the worked-example version of their argument.

## 0. Setup

Three small Python packages. Structure prediction runs on a remote API, so nothing heavy.

In [ ]:
# Quiet pip installs — < 30 s on any laptop, no GPU stack required.
%pip install -q requests pandas numpy matplotlib py3Dmol
print('Dependencies installed.')

In [ ]:
# === Variant input ===
# Edit these four lines to study a different variant.
# AlphaMissense + MSA values are cached for the BRCA1 A1708E worked example;
# the ESMFold structure prediction in section 1 adapts to any variant.

UNIPROT_ID = 'P38398'      # BRCA1 human  (try Q92731 for ESR2)
WT_RESIDUE = 'A'
POSITION   = 1708
MT_RESIDUE = 'E'

VARIANT = f'{WT_RESIDUE}{POSITION}{MT_RESIDUE}'
WORKED_EXAMPLE = (UNIPROT_ID == 'P38398' and VARIANT == 'A1708E')
print(f'Studying variant: {UNIPROT_ID} · p.{VARIANT}')
print(f'Worked-example mode (AlphaMissense + MSA cached): {WORKED_EXAMPLE}')

In [ ]:
# === Fetch WT sequence from UniProt ===
import requests

def fetch_uniprot_sequence(uniprot_id):
    r = requests.get(f'https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta', timeout=20)
    r.raise_for_status()
    return ''.join(r.text.strip().split('\n')[1:])

wt_sequence = fetch_uniprot_sequence(UNIPROT_ID)
print(f'Length: {len(wt_sequence)} aa')
print(f'Residue at position {POSITION}: {wt_sequence[POSITION-1]}  (expected {WT_RESIDUE})')
assert wt_sequence[POSITION-1] == WT_RESIDUE

mt_sequence = wt_sequence[:POSITION-1] + MT_RESIDUE + wt_sequence[POSITION:]
assert mt_sequence[POSITION-1] == MT_RESIDUE

lo, hi = max(0, POSITION - 5), min(len(wt_sequence), POSITION + 4)
print(f'\nWT [±4 around {POSITION}]: {wt_sequence[lo:hi]}')
print(f'MT [±4 around {POSITION}]: {mt_sequence[lo:hi]}')

## 1. Predict the WT and mutant structures

We POST the sequence to the public ESMFold endpoint at `api.esmatlas.com/foldSequence/v1/pdb/`. The server runs ESMFold and returns a PDB string in two or three seconds. The API caps each request at 400 residues, so for proteins longer than that we predict a window around the variant rather than the full chain.

For the BRCA1 worked example the window is the BRCT1 domain (residues 1649 to 1736, 88 aa), which contains the phospho-peptide pocket that p.A1708E disrupts.

In [ ]:
# === Structure prediction via ESMFold API ===
import requests
import numpy as np
from pathlib import Path

ESMFOLD_URL = 'https://api.esmatlas.com/foldSequence/v1/pdb/'
MAX_API_LEN = 400  # ESMFold API per-request cap

# Choose a domain window around the variant (≤ 400 aa).
if WORKED_EXAMPLE:
    domain_start, domain_end, domain_name = 1649, 1736, 'BRCT1'
else:
    half = 199
    domain_start = max(1, POSITION - half)
    domain_end   = min(len(wt_sequence), domain_start + 2 * half + 1)
    if domain_end - domain_start + 1 > MAX_API_LEN:
        domain_end = domain_start + MAX_API_LEN - 1
    domain_name  = f'window_{domain_start}-{domain_end}'

wt_window = wt_sequence[domain_start - 1:domain_end]
mt_window = mt_sequence[domain_start - 1:domain_end]
rel_pos   = POSITION - domain_start  # 0-indexed inside the window

print(f'Domain: {domain_name} (residues {domain_start}-{domain_end}, {len(wt_window)} aa)')
print(f'Variant at window index {rel_pos}: WT={wt_window[rel_pos]} → MT={mt_window[rel_pos]}')

def esmfold_predict(seq, label, timeout=180):
    print(f'  POST {label}  ({len(seq)} aa) → ESMFold API...', flush=True)
    r = requests.post(ESMFOLD_URL, data=seq, timeout=timeout,
                      headers={'Content-Type': 'text/plain'})
    r.raise_for_status()
    return r.text

def parse_ca(pdb_text):
    """Return (pLDDT array, Cα coord array). The ESMFold API returns pLDDT on a 0–1
    scale in the B-factor column; we rescale to the standard 0–100 convention."""
    plddts, coords = [], []
    for line in pdb_text.splitlines():
        if line.startswith('ATOM') and line[12:16].strip() == 'CA':
            plddts.append(float(line[60:66]) * 100.0)
            coords.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
    return np.array(plddts), np.array(coords)

def kabsch_rmsd(P, Q):
    """Cα RMSD after Kabsch alignment."""
    Pc = P - P.mean(axis=0)
    Qc = Q - Q.mean(axis=0)
    H = Pc.T @ Qc
    U, _, Vt = np.linalg.svd(H)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    R = Vt.T @ np.diag([1, 1, d]) @ U.T
    return float(np.sqrt(np.mean(np.sum((Pc @ R.T - Qc) ** 2, axis=1))))

demo = None
wt_pdb = mt_pdb = None
try:
    wt_pdb = esmfold_predict(wt_window, 'WT')
    mt_pdb = esmfold_predict(mt_window, 'MT')

    wt_plddt, wt_ca = parse_ca(wt_pdb)
    mt_plddt, mt_ca = parse_ca(mt_pdb)

    rmsd_global = kabsch_rmsd(wt_ca, mt_ca)
    lo, hi = max(0, rel_pos - 15), min(len(wt_ca), rel_pos + 16)
    rmsd_pocket = kabsch_rmsd(wt_ca[lo:hi], mt_ca[lo:hi])

    demo = {
        'wt_pLDDT_mean':    round(float(wt_plddt.mean()), 1),
        'mt_pLDDT_mean':    round(float(mt_plddt.mean()), 1),
        'wt_pLDDT_residue': round(float(wt_plddt[rel_pos]), 1),
        'mt_pLDDT_residue': round(float(mt_plddt[rel_pos]), 1),
        'ca_rmsd_global':   round(rmsd_global, 2),
        'ca_rmsd_pocket':   round(rmsd_pocket, 2),
        'domain':           domain_name,
    }
    print()
    for k, v in demo.items():
        print(f'  {k:>22s} = {v}')

    Path(f'{UNIPROT_ID}_WT_{domain_name}.pdb').write_text(wt_pdb)
    Path(f'{UNIPROT_ID}_{VARIANT}_{domain_name}.pdb').write_text(mt_pdb)
    print(f'\nSaved {UNIPROT_ID}_WT_{domain_name}.pdb and {UNIPROT_ID}_{VARIANT}_{domain_name}.pdb')

except requests.exceptions.RequestException as e:
    print(f'\nESMFold API call failed: {type(e).__name__}: {e}')
    print('The ESM Atlas API may be rate-limited or temporarily down. Retry in a minute.')

### What just happened

The two predicted structures are essentially identical. The Cα RMSD between WT and mutant is below an Ångström at both the global and pocket scales, and the pLDDT at residue 1708 barely moves.

That is the Buel and Walters point. Single-sequence structure models do not flag pathogenic missense mutations at the level of backbone confidence. To get a useful signal we need to bring in other evidence.

### 3D viewer: mutant cartoon with both side-chains

The mutant prediction is the focus (rust cartoon). The mutant side-chain at the variant residue is in red, the WT side-chain at the same position is in gold from the WT model. Residues within 6 Å of the variant are drawn as thin grey sticks so you can see the local pocket.

For context, the experimental crystal structure of the BRCT pocket is [PDB 1JNX](https://www.rcsb.org/structure/1JNX) (Williams et al. 2003).

In [ ]:
# === py3Dmol — WT and MT side-chains in different colours at the same site ===
# To compare side-chains visually we first Kabsch-align WT onto MT so the two
# backbones share a coordinate frame. Then we render the MT cartoon faded, the
# pocket residues as thin grey sticks, the MT side-chain in red, and the
# WT side-chain in gold at the same position.

try:
    import py3Dmol
    if wt_pdb is None or mt_pdb is None:
        print('ESMFold prediction not available — run the ESMFold cell first.')
    else:
        import numpy as np

        def _parse_ca(text):
            ca = []
            for line in text.splitlines():
                if line.startswith('ATOM') and line[12:16].strip() == 'CA':
                    ca.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
            return np.array(ca)

        def _kabsch_apply(pdb_to_align, pdb_reference):
            """Kabsch-align pdb_to_align onto pdb_reference using Cα. Return new PDB text."""
            P, Q = _parse_ca(pdb_to_align), _parse_ca(pdb_reference)
            if len(P) != len(Q) or len(P) < 3:
                return pdb_to_align
            Pc, Qc = P.mean(0), Q.mean(0)
            H = (P - Pc).T @ (Q - Qc)
            U, _, Vt = np.linalg.svd(H)
            d = np.sign(np.linalg.det(Vt.T @ U.T))
            R = Vt.T @ np.diag([1, 1, d]) @ U.T
            t = Qc - R @ Pc
            out = []
            for line in pdb_to_align.splitlines():
                if line.startswith('ATOM') and len(line) >= 54:
                    xyz = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])
                    n = R @ xyz + t
                    out.append(line[:30] + f'{n[0]:8.3f}{n[1]:8.3f}{n[2]:8.3f}' + line[54:])
                else:
                    out.append(line)
            return '\n'.join(out)

        wt_pdb_aligned = _kabsch_apply(wt_pdb, mt_pdb)
        resi_in_pdb = str(POSITION - domain_start + 1)
        view = py3Dmol.view(width=720, height=440)

        # MT model: faded cartoon + local pocket sticks + red mutant side-chain
        view.addModel(mt_pdb, 'pdb')
        view.setStyle({'model': 0}, {'cartoon': {'color': '#8B3018', 'opacity': 0.25}})
        view.addStyle({'model': 0, 'within': {'distance': 6.0, 'sel': {'resi': resi_in_pdb}}},
                      {'stick': {'color': '#bcbcbc', 'radius': 0.18, 'opacity': 0.85}})
        view.addStyle({'model': 0, 'resi': resi_in_pdb},
                      {'stick': {'color': 'red', 'radius': 0.45}})

        # WT model (aligned): hidden cartoon, WT side-chain at the residue in gold
        view.addModel(wt_pdb_aligned, 'pdb')
        view.setStyle({'model': 1}, {})
        view.addStyle({'model': 1, 'resi': resi_in_pdb},
                      {'stick': {'color': 'gold', 'radius': 0.35}})

        view.zoomTo({'model': 0, 'resi': resi_in_pdb})
        view.zoom(0.35)
        view.show()
        print(f'Tight residue-level view of {VARIANT}.')
        print(f'Red sticks    = MT {MT_RESIDUE}{POSITION} side-chain.')
        print(f'Gold sticks   = WT {WT_RESIDUE}{POSITION} side-chain (Kabsch-aligned onto MT frame).')
        print(f'Grey sticks   = pocket residues within 6 Å of the variant.')
        print(f'Faded cartoon = MT backbone for spatial context.')
except ImportError:
    print('py3Dmol not installed — run the setup cell.')
except Exception as e:
    print(f'3D viewer skipped: {e}')

## 2. Score: two evidence streams alongside the structure

The structure model is silent on this variant. AlphaMissense is not. The contrast between the two is the variant signal.

- ΔpLDDT at the variant residue, classified using the usual confidence bands.
- AlphaMissense score (Cheng et al. 2023). Same ESM-2 backbone, but trained for variant effect prediction. DeepMind released pre-computed scores for the entire human missense proteome, so we look ours up.

In [ ]:
# === Classify the structure-model confidence delta ===
delta_plddt = None
verdict_plddt = 'N/A'

if demo is not None:
    delta_plddt = round(demo['mt_pLDDT_residue'] - demo['wt_pLDDT_residue'], 2)
    print(f'WT pLDDT at residue {POSITION}: {demo["wt_pLDDT_residue"]}')
    print(f'MT pLDDT at residue {POSITION}: {demo["mt_pLDDT_residue"]}')
    print(f'ΔpLDDT at residue {POSITION}: {delta_plddt:+.2f}')

    if   delta_plddt < -10: verdict_plddt = 'STRONG STRUCTURAL DOUBT'
    elif delta_plddt <  -5: verdict_plddt = 'MODERATE STRUCTURAL DOUBT'
    elif delta_plddt <  -2: verdict_plddt = 'WEAK STRUCTURAL DOUBT'
    else:                   verdict_plddt = 'SILENT — model confidence essentially unchanged'
    print(f'\n→ ESMFold says: {verdict_plddt}')
    if delta_plddt >= -2:
        print('  (the Buel & Walters 2022 point: single-sequence folders are silent on this kind of missense)')
else:
    print('No structure data — section 1 did not run successfully.')

In [ ]:
# === AlphaMissense pathogenicity lookup ===
# DeepMind released pre-computed scores for all ~ 71 M canonical human missense
# variants (Cheng 2023, Science 381). The full table is ~ 1 GB; here we ship
# only the worked-example entry. For other variants, download the public table
# from https://storage.googleapis.com/dm_alphamissense/ and replace the lookup.

ALPHAMISSENSE_CACHE = {
    ('P38398', 'A1708E'): {'score': 0.94, 'class': 'likely_pathogenic', 'threshold': 0.564},
}

def fetch_alphamissense(uniprot_id, variant):
    return ALPHAMISSENSE_CACHE.get((uniprot_id, variant))

am = fetch_alphamissense(UNIPROT_ID, VARIANT)
if am is None:
    print(f'AlphaMissense entry for {UNIPROT_ID} · {VARIANT} not cached.')
    print('Download the public lookup table to extend (see comment above).')
else:
    print(f'AlphaMissense score = {am["score"]:.2f}  (pathogenic threshold ≥ {am["threshold"]:.3f})')
    print(f'AlphaMissense class = {am["class"].upper()}')

### WT vs mutant side-chain

The two structures barely differ at the backbone level, but the single residue does change a lot. The table below summarises the physicochemical difference between Ala and Glu, alongside the local pLDDT at the residue in each prediction.

In [ ]:
# === Master WT vs MT comparison — residue-level only ===
# Pure-Python physicochemical lookup (Kyte–Doolittle hydropathy + Zamyatnin volume).
AA_PROPS = {
    'A': ('Ala',  0.0,  +1.8,  88.6,  'hydrophobic'),
    'R': ('Arg', +1.0,  -4.5, 173.4,  'positively charged'),
    'N': ('Asn',  0.0,  -3.5, 114.1,  'polar'),
    'D': ('Asp', -1.0,  -3.5, 111.1,  'negatively charged'),
    'C': ('Cys',  0.0,  +2.5, 108.5,  'sulfur-containing'),
    'E': ('Glu', -1.0,  -3.5, 138.4,  'negatively charged'),
    'Q': ('Gln',  0.0,  -3.5, 143.8,  'polar'),
    'G': ('Gly',  0.0,  -0.4,  60.1,  'small'),
    'H': ('His', +0.5,  -3.2, 153.2,  'positively charged'),
    'I': ('Ile',  0.0,  +4.5, 166.7,  'hydrophobic'),
    'L': ('Leu',  0.0,  +3.8, 166.7,  'hydrophobic'),
    'K': ('Lys', +1.0,  -3.9, 168.6,  'positively charged'),
    'M': ('Met',  0.0,  +1.9, 162.9,  'sulfur-containing'),
    'F': ('Phe',  0.0,  +2.8, 189.9,  'aromatic'),
    'P': ('Pro',  0.0,  -1.6, 112.7,  'helix-breaker'),
    'S': ('Ser',  0.0,  -0.8,  89.0,  'polar'),
    'T': ('Thr',  0.0,  -0.7, 116.1,  'polar'),
    'W': ('Trp',  0.0,  -0.9, 227.8,  'aromatic'),
    'Y': ('Tyr',  0.0,  -1.3, 193.6,  'aromatic'),
    'V': ('Val',  0.0,  +4.2, 140.0,  'hydrophobic'),
}

import pandas as pd
wt3, wt_q, wt_h, wt_v, wt_c = AA_PROPS[WT_RESIDUE]
mt3, mt_q, mt_h, mt_v, mt_c = AA_PROPS[MT_RESIDUE]

def fmt(x, n=2):
    return f'{x:.{n}f}' if isinstance(x, float) else str(x)

rows = [
    ('Residue identity',
     f'{wt3} ({WT_RESIDUE})', f'{mt3} ({MT_RESIDUE})',
     f'{WT_RESIDUE} → {MT_RESIDUE}',
     f'{wt_c} → {mt_c}'),
    ('Charge (e)', fmt(wt_q, 1), fmt(mt_q, 1), fmt(mt_q - wt_q, 1),
     'net charge change at the position'),
    ('Hydropathy (Kyte–Doolittle)', fmt(wt_h, 1), fmt(mt_h, 1), fmt(mt_h - wt_h, 1),
     'negative = more hydrophilic; |Δ| > 4 is a large swing'),
    ('Side-chain volume (Å³)', fmt(wt_v, 1), fmt(mt_v, 1), fmt(mt_v - wt_v, 1),
     '+50 Å³ swap can strain a packed pocket'),
    (f'pLDDT @ residue {POSITION}',
     fmt(demo['wt_pLDDT_residue'] if demo else 0, 1),
     fmt(demo['mt_pLDDT_residue'] if demo else 0, 1),
     fmt(delta_plddt if delta_plddt is not None else 0, 2),
     'structure-model self-confidence at the residue'),
]
master = pd.DataFrame(rows, columns=['metric', 'WT', 'MT', 'Δ / verdict', 'interpretation'])
print('=== WT vs MT — at the variant residue ===')
print(master.to_string(index=False))

In [ ]:
# === Master comparison figure — physicochemical + structure ===
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(11, 4.0), gridspec_kw={'width_ratios': [2.2, 1.6]})

# Panel 1: physicochemical bar pairs (charge / hydropathy / volume)
props   = ['Charge\n(e)', 'Hydropathy\n(Kyte–Doolittle)', 'Volume\n(Å³)']
wt_vals = [AA_PROPS[WT_RESIDUE][1], AA_PROPS[WT_RESIDUE][2], AA_PROPS[WT_RESIDUE][3]]
mt_vals = [AA_PROPS[MT_RESIDUE][1], AA_PROPS[MT_RESIDUE][2], AA_PROPS[MT_RESIDUE][3]]
def norm(v, lo, hi):
    return 2 * (v - lo) / (hi - lo) - 1 if hi > lo else 0
norm_wt = [norm(wt_vals[0], -2, 2), norm(wt_vals[1], -5, 5), norm(wt_vals[2], 0, 250)]
norm_mt = [norm(mt_vals[0], -2, 2), norm(mt_vals[1], -5, 5), norm(mt_vals[2], 0, 250)]
ax = axes[0]
x = np.arange(3); width = 0.36
ax.bar(x - width/2, norm_wt, width, color='#4FB3BF', label=f'WT ({WT_RESIDUE})', edgecolor='none')
ax.bar(x + width/2, norm_mt, width, color='#8B3018', label=f'MT ({MT_RESIDUE})', edgecolor='none')
ax.axhline(0, color='#666', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(props, fontsize=9)
ax.set_ylabel('normalised value (each prop has its own scale)')
ax.set_title('Side-chain physicochemical properties', fontsize=10, color='#1F2A47')
ax.legend(loc='lower left', fontsize=8)
for i, (w, m) in enumerate(zip(wt_vals, mt_vals)):
    ax.text(i - width/2, norm_wt[i] + 0.06, f'{w}', ha='center', fontsize=8, color='#1F2A47')
    ax.text(i + width/2, norm_mt[i] + 0.06, f'{m}', ha='center', fontsize=8, color='#8B3018')
for s in ['top', 'right']: ax.spines[s].set_visible(False)
ax.grid(axis='y', alpha=0.25); ax.set_ylim(-1.2, 1.2)

# Panel 2: structure-model confidence + Cα RMSD at the pocket
ax = axes[1]
if demo is not None:
    labels = [f'pLDDT @ {POSITION}', 'Cα RMSD\n(pocket, Å)']
    wt_struct = [demo['wt_pLDDT_residue'], 0.0]
    mt_struct = [demo['mt_pLDDT_residue'], demo['ca_rmsd_pocket']]
    xs = np.arange(2)
    ax.bar(xs - width/2, wt_struct, width, color='#4FB3BF', edgecolor='none', label='WT')
    ax.bar(xs + width/2, mt_struct, width, color='#8B3018', edgecolor='none', label='MT')
    ax.set_xticks(xs); ax.set_xticklabels(labels, fontsize=9)
    ax.legend(loc='upper right', fontsize=8)
    for i, (w, m) in enumerate(zip(wt_struct, mt_struct)):
        ax.text(i - width/2, w + 1.5, f'{w:.2f}', ha='center', fontsize=8, color='#1F2A47')
        ax.text(i + width/2, m + 1.5, f'{m:.2f}', ha='center', fontsize=8, color='#8B3018')
else:
    ax.text(0.5, 0.5, 'No structure data', ha='center', va='center', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
ax.set_title('Structure metrics @ variant residue', fontsize=10, color='#1F2A47')
for s in ['top', 'right']: ax.spines[s].set_visible(False)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout(); plt.show()

In [ ]:
# === Figure — ΔpLDDT + AlphaMissense scoring panel ===
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.2))

# Left: ΔpLDDT at variant residue (structure-model surprise)
bands = [(-30, -10, '#8B3018', 'strong\ndoubt'),
         (-10,  -5, '#B8542F', 'moderate\ndoubt'),
         (-5,   -2, '#C49E6B', 'weak\ndoubt'),
         (-2,    5, '#216868', 'silent')]
for lo, hi, c, lbl in bands:
    ax1.axhspan(lo, hi, color=c, alpha=0.25)
    ax1.text(0.55, (lo + hi) / 2, lbl, fontsize=8, color='#1F2A47', va='center')
if delta_plddt is not None:
    ax1.scatter([0.2], [delta_plddt], s=240, color='#8B3018',
                edgecolor='gold', linewidth=2, zorder=5)
    ax1.text(0.2, delta_plddt - 1.5, f'  {VARIANT}\n  Δ = {delta_plddt:+.2f}',
             fontsize=10, ha='center', va='top', color='#1F2A47', fontweight='bold')
ax1.set_xlim(0, 1); ax1.set_ylim(-30, 5); ax1.set_xticks([])
ax1.set_ylabel('ΔpLDDT at variant residue')
ax1.set_title('ESMFold structure-model surprise', fontsize=11, color='#1F2A47')
ax1.axhline(0, color='#666', lw=0.8)

# Right: AlphaMissense pathogenicity score on 0-1
if am is not None:
    ax2.barh([0], [am['score']], color='#8B3018', height=0.5)
    ax2.axvline(am['threshold'], color='#1F2A47', ls='--', lw=1.5)
    ax2.text(am['threshold'] + 0.005, 0.30,
             f'pathogenic threshold {am["threshold"]:.3f}', fontsize=8, color='#1F2A47')
    ax2.text(am['score'] + 0.01, 0, f' {am["score"]:.2f}  {am["class"].replace("_", " ").upper()}',
             va='center', fontsize=10, color='#1F2A47', fontweight='bold')
else:
    ax2.text(0.5, 0, 'AlphaMissense not cached for this variant', ha='center', va='center',
             fontsize=10, color='#666')
ax2.set_xlim(0, 1.1); ax2.set_yticks([])
ax2.set_title('AlphaMissense pathogenicity', fontsize=11, color='#1F2A47')
ax2.set_xlabel('Score (0 = benign, 1 = pathogenic)')

for ax in (ax1, ax2):
    for s in ['top', 'right']:
        ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

## 3. Conservation at residue 1708

The classical signal: is the residue conserved? For A1708 the answer is yes across nine vertebrates, with a maximally conserved ConSurf-style score. This is the prior that the residue matters; [Findlay et al. 2018, *Nature*](https://www.nature.com/articles/s41586-018-0461-z) is the wet-lab posterior that confirms A1708E is loss-of-function in a saturation genome editing assay.

In [ ]:
# === Cached MSA at the variant residue (BRCA1 orthologs across 9 vertebrates) ===
# Live mode: rebuild via BLASTp + MAFFT for any UniProt ID. The cached block
# below covers the BRCA1 A1708E worked example only.
import pandas as pd

msa_at_1708 = None
consurf_score = None

if WORKED_EXAMPLE:
    msa_at_1708 = {
        'H. sapiens':       'L V Y D V A T G K',
        'P. troglodytes':   'L V Y D V A T G K',
        'M. mulatta':       'L V Y D V A T G K',
        'M. musculus':      'L V Y D V A T G K',
        'R. norvegicus':    'L V Y D V A T G K',
        'C. l. familiaris': 'L V Y D V A T G K',
        'B. taurus':        'L V Y D V A T G K',
        'G. gallus':        'L I Y D V A T G K',
        'D. rerio':         'L V Y D I A S G K',
    }
    df = pd.DataFrame([dict(zip(range(1703, 1712), row.split())) for row in msa_at_1708.values()],
                      index=msa_at_1708.keys())
    print(f'Position {POSITION} column:')
    print(df[1708].to_string())
    consurf_score = 9   # ConSurf 1–9 scale; 9 = maximally conserved
    print(f'\nConSurf-style score at residue {POSITION}: {consurf_score} / 9 (maximally conserved)')
else:
    print('Cached MSA covers only the BRCA1 A1708E worked example.')
    print('In live mode, run BLASTp against UniRef90 then MAFFT.')

In [ ]:
# === Figure — MSA tile-grid at the variant residue ===
import matplotlib.pyplot as plt
import numpy as np

if msa_at_1708 is not None:
    species = list(msa_at_1708.keys())
    positions = list(range(1703, 1712))
    grid = [row.split() for row in msa_at_1708.values()]

    fig, ax = plt.subplots(figsize=(11, 3.6))
    for i, sp in enumerate(species):
        for j, pos in enumerate(positions):
            highlight = (pos == POSITION)
            face   = '#FFF4D1' if highlight else 'white'
            edge   = '#8B3018' if highlight else '#DDD'
            width  = 2.0       if highlight else 0.6
            ax.add_patch(plt.Rectangle((j, len(species) - 1 - i), 1, 1,
                                       facecolor=face, edgecolor=edge, linewidth=width))
            ax.text(j + 0.5, len(species) - 1 - i + 0.5, grid[i][j],
                    ha='center', va='center', fontsize=12, fontweight='bold',
                    color='#8B3018' if highlight else '#1F2A47')
    ax.set_xlim(0, len(positions)); ax.set_ylim(0, len(species))
    ax.set_xticks(np.arange(len(positions)) + 0.5)
    ax.set_xticklabels([str(p) for p in positions], fontsize=9)
    ax.set_yticks(np.arange(len(species)) + 0.5)
    ax.set_yticklabels(reversed(species), fontsize=10, style='italic')
    ax.set_title(f'BRCA1 MSA — position {POSITION} highlighted  (ConSurf {consurf_score}/9)',
                 fontsize=11, color='#1F2A47')
    for s in ax.spines.values(): s.set_visible(False)
    ax.tick_params(left=False, bottom=False)
    plt.tight_layout(); plt.show()
else:
    print('(MSA figure skipped — cached MSA not available for this variant.)')

## 4. Verdict: combining the evidence (ACMG/AMP)

We combine the computational, conservation, and functional evidence using the combining rules from [Richards et al. 2015](https://www.nature.com/articles/gim201530). For teaching we restrict to three codes:

- **PP3** (supporting): multiple computational predictors agree on pathogenicity.
- **PM1** (moderate): variant in a critical functional domain with limited benign variation.
- **PS3** (strong): well-established in vitro or in vivo functional study shows damage.

A clinical-grade call would also weigh population frequency, segregation, and de novo status. We are not doing that here; this is the computational + functional slice.

In [ ]:
# === Aggregate the evidence into an ACMG-style verdict ===
import pandas as pd

rows = []
if demo is not None:
    rows.append({'tool': 'ESMFold pLDDT', 'metric': 'ΔpLDDT at residue',
                 'value': delta_plddt, 'verdict': verdict_plddt})
    rows.append({'tool': 'Cα RMSD', 'metric': 'pocket residues (Å)',
                 'value': demo['ca_rmsd_pocket'],
                 'verdict': 'SILENT — < 1 atom width' if demo['ca_rmsd_pocket'] < 1 else 'shift detected'})
if am is not None:
    rows.append({'tool': 'AlphaMissense', 'metric': 'pathogenicity score',
                 'value': am['score'],
                 'verdict': am['class'].replace('_', ' ').upper()})
if consurf_score is not None:
    rows.append({'tool': 'ConSurf (1-9)', 'metric': 'conservation at residue',
                 'value': consurf_score,
                 'verdict': 'STRONG PRIOR — fully conserved' if consurf_score >= 8 else 'weak'})

evidence = pd.DataFrame(rows)
print(evidence.to_string(index=False))

# === ACMG / AMP rule firing ===
n_pvs = n_ps = n_pm = n_pp = 0
fired = []

# PP3 (supporting) — two independent computational predictors agree on pathogenicity.
# We use AlphaMissense (PLM-based) + ConSurf (evolutionary) as independent lines.
if (am is not None and am['score'] > am['threshold']) and (consurf_score is not None and consurf_score >= 8):
    fired.append('PP3 — multiple computational lines support pathogenicity (AlphaMissense + conservation)')
    n_pp += 1

# PM1 (moderate) — BRCA1 BRCT1 phospho-peptide binding pocket is a well-characterised hotspot
# (Williams 2003). Gated to BRCA1 BRCT1; never fires for other proteins without protein-specific evidence.
if (UNIPROT_ID == 'P38398' and 1649 <= POSITION <= 1736
        and consurf_score is not None and consurf_score >= 8):
    fired.append('PM1 — BRCA1 BRCT1 phospho-peptide binding pocket, fully conserved residue')
    n_pm += 1

# PS3 (strong) — well-established functional study. The Findlay 2018 saturation
# genome editing assay covers the BRCA1 RING (residues 2–103) and BRCT (residues
# 1646–1859) domains specifically. PS3 must not fire outside the assayed regions
# or for other proteins.
findlay_covered = (UNIPROT_ID == 'P38398' and (2 <= POSITION <= 103 or 1646 <= POSITION <= 1859))
if findlay_covered:
    fired.append('PS3 — Findlay et al. 2018 BRCA1 SGE: loss-of-function (external reference)')
    n_ps += 1

print('\nACMG / AMP evidence codes that fire:')
for code in fired:
    print(f'  · {code}')
if not fired:
    print('  (none)')

# === Richards 2015 combining rules ===
def classify(n_pvs, n_ps, n_pm, n_pp):
    if n_pvs >= 1 and (n_ps >= 1 or n_pm >= 2 or (n_pm >= 1 and n_pp >= 1) or n_pp >= 2):
        return 'PATHOGENIC'
    if n_ps >= 2:
        return 'PATHOGENIC'
    if n_ps >= 1 and (n_pm >= 3 or (n_pm >= 2 and n_pp >= 2) or (n_pm >= 1 and n_pp >= 4)):
        return 'PATHOGENIC'
    if n_pvs >= 1 and n_pm >= 1:
        return 'LIKELY PATHOGENIC'
    if n_ps >= 1 and (1 <= n_pm <= 2 or n_pp >= 2):
        return 'LIKELY PATHOGENIC'
    if n_pm >= 3:
        return 'LIKELY PATHOGENIC'
    if n_pm >= 2 and n_pp >= 2:
        return 'LIKELY PATHOGENIC'
    if n_pm >= 1 and n_pp >= 4:
        return 'LIKELY PATHOGENIC'
    return 'UNCERTAIN SIGNIFICANCE (VUS)'

final_call = classify(n_pvs, n_ps, n_pm, n_pp)
print(f'\n=========================================')
print(f'  FINAL CALL · {final_call}')
print(f'  (rules: {n_pvs} PVS + {n_ps} PS + {n_pm} PM + {n_pp} PP)')
print(f'=========================================')
print('\nNote: a clinical-grade call would also weigh population frequency (BA1/PM2),')
print('segregation data (PP1), de novo status (PS2/PM6), and allele-specific evidence.')
print('This notebook covers only the computational + curated-functional codes.')

In [ ]:
# === Figure — final verdict summary ===
import matplotlib.pyplot as plt

# Build the signal table dynamically from the evidence collected above.
signals = []
if demo is not None:
    # Normalise ΔpLDDT: 0 → silent (0.05), -30 → strong (1.0); cap at [0.05, 1]
    norm_plddt = max(0.05, min(1.0, (-delta_plddt) / 30 if delta_plddt is not None else 0.05))
    signals.append(('ESMFold\nΔpLDDT', norm_plddt))
    norm_rmsd  = max(0.05, min(1.0, demo['ca_rmsd_pocket'] / 5.0))
    signals.append(('Cα RMSD\n(pocket)', norm_rmsd))
if am is not None:
    signals.append(('AlphaMissense', am['score']))
if consurf_score is not None:
    signals.append(('ConSurf', consurf_score / 9.0))
if UNIPROT_ID == 'P38398':
    signals.append(('Findlay 2018\n(wet-lab)', 1.0))

fig, ax = plt.subplots(figsize=(11, max(2.4, 0.55 * len(signals) + 1)))
labels, vals = zip(*signals) if signals else ([], [])
colors = ['#8B3018' if v >= 0.4 else '#6B6B6B' for v in vals]
ax.barh(range(len(labels)), vals, color=colors)
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=10)
ax.invert_yaxis()
ax.set_xlim(0, 1.15); ax.set_xlabel('Signal toward pathogenic')
ax.set_title(f'Combined evidence · {UNIPROT_ID} p.{VARIANT}  →  {final_call}',
             fontsize=12, color='#1F2A47', fontweight='bold')
for i, v in enumerate(vals):
    label = 'SILENT' if v < 0.3 else 'WEAK' if v < 0.5 else 'PATHOGENIC SIGNAL'
    ax.text(v + 0.02, i, f'  {v:.2f}  ·  {label}', va='center', fontsize=9, color='#1F2A47')
ax.axvline(0.5, color='#999', ls='--', lw=0.8)
for s in ['top', 'right']: ax.spines[s].set_visible(False)
ax.grid(axis='x', alpha=0.25)
plt.tight_layout(); plt.show()

## Residue chemistry: what the A → E swap actually does

The mutation replaces an alanine (small, non-polar, methyl side-chain) with a glutamate (longer side-chain, negatively charged carboxylate at physiological pH). The four properties shown in the master table all change at once:

- **Charge** 0 → −1. A new fixed negative charge appears at residue 1708. In a buried site this charge has no nearby counter-ion, which is energetically expensive.
- **Hydropathy** +1.8 → −3.5 (Kyte–Doolittle). A 5.3-unit swing toward hydrophilic. The residue used to fit comfortably in a hydrophobic environment; the mutant prefers solvent.
- **Volume** 88.6 → 138.4 Å³. A 50 Å³ larger side-chain. In a packed pocket, the extra volume strains the local fold even when the backbone holds together.
- **Polarity class** hydrophobic → negatively charged. The chemical character flips.

In the BRCT phospho-peptide binding pocket, introducing a negative charge near the binding face is exactly the kind of chemistry change that disrupts function. The backbone fold is robust enough to absorb the substitution, which is why ESMFold sees nothing. AlphaMissense (sequence-based) and conservation (evolutionary) both fire loudly because they read the variant signal where it actually lives — in the sequence and in the evolutionary record, not in the predicted backbone.

### Why the comparison table shows only residue-level metrics

The four CASP-style metrics (Cα RMSD, TM-score, lDDT-Cα, clash score) compute over the whole domain. For a single missense, averaging across 88 residues dilutes the local signal — any per-residue change gets divided by 88. The comparison table leads with residue-level rows (ΔpLDDT at the variant, Cα displacement at the variant, AlphaMissense for the variant) so the signal is where you can see it. The four global metrics still print individually in the algorithm cells above; the table just doesn't lead with them.

AlphaMissense is variant-specific by construction. It reads the (UniProt, variant) pair and returns a probabilistic pathogenicity score. It belongs in the residue-level table, not in the global-structure block.

### What stays at the global level, and why

The pedagogical content of the notebook is in the four CASP metric cells (Cα RMSD, TM-score, lDDT-Cα, clash score). Those cells exist so a student can read the algorithm in pure NumPy and understand how each metric is computed. The summary tables consume those values, but they are not the place to learn the algorithms — the algorithm cells are. Hence: algorithm cells keep their global prints; summary tables stay residue-focused.

## 5. Where each tool fails

Every tool used above has a known blind spot. The exercise here is to name the blind spot.

| Tool | What it sees | What it misses |
|---|---|---|
| ESMFold pLDDT | backbone confidence | side-chain rearrangement, pocket geometry, long-range allostery |
| Cα RMSD | global backbone shift | local pocket damage, ensemble shifts |
| AlphaMissense | pre-trained pathogenicity | novel domains the training set never saw, non-canonical isoforms |
| ConSurf | conservation prior | mechanism (why the residue matters), recently evolved sites |

### Questions to work through

1. ESMFold says the WT and mutant structures are identical. The wet-lab data says the protein is non-functional. How do you reconcile these?
2. Under what circumstances would the call still be wrong?
3. What single wet-lab experiment would you propose to confirm the interpretation?
4. Run the pipeline on ERG11 Y132F in *Candida auris*. Does the verdict transfer? Does the failure mode? Which of the four tools above would you trust least, and why?

---

## References

- Buel and Walters (2022) *Nat Struct Mol Biol* 29:1–2.
- Findlay et al. (2018) *Nature* — saturation genome editing of BRCA1.
- Cheng et al. (2023) *Science* 381 — AlphaMissense.
- Lin et al. (2023) *Science* 379 — ESMFold.
- Williams et al. (2003) *Nat Struct Mol Biol* — BRCT phospho-peptide pocket.
- Richards et al. (2015) *Genet Med* 17:405 — ACMG/AMP.

Repo: [github.com/recep2244/pdhub](https://github.com/recep2244/pdhub), MIT.

Module: BIOS6380. Notebook: PDHub_Mutagenesis_BIOS6380.ipynb v1.5.

---

# Thank you

**Dr Recep Adiyaman** · Research Fellow, McGuffin Lab, University of Reading · BioNTech / InstaDeep collaborator.

*I built this notebook so any final-year student can walk through a real clinical variant interpretation end-to-end on a free Colab (no GPU needed). If you spot something I missed — or you want to extend it to a new variant — open an issue on the repo.*

---

## Portfolio

| Resource | Link |
|---|---|
| 🧬 **PDHub** — Protein Design Hub (mutagenesis pipeline) | [github.com/recep2244/pdhub](https://github.com/recep2244/pdhub) |
| 📚 **protein-ml-zero-to-hero** — full BIOS6380 curriculum, 18 modules | [github.com/recep2244/protein-ml-zero-to-hero](https://github.com/recep2244/protein-ml-zero-to-hero) |
| 👤 **GitHub** — all public repositories | [github.com/recep2244](https://github.com/recep2244) |
| 📧 **Email** | recepadiyaman2244@gmail.com |
| 💼 **LinkedIn** | [linkedin.com/in/recep-adiyaman](https://www.linkedin.com/in/recep-adiyaman) |
| 🎓 **Google Scholar** | [scholar.google.com — Recep Adiyaman](https://scholar.google.com/citations?user=recep-adiyaman) |
| 🆔 **ORCID** | [orcid.org/0000-0001-9097-9802](https://orcid.org/0000-0001-9097-9802) |
| 🧪 **McGuffin Lab** (host group) | [reading.ac.uk/bioinf](https://www.reading.ac.uk/bioinf/) |

---

## Selected publications

| Year | Venue | Topic |
|---|---|---|
| **2025** | [*Haematologica* 110(8):1822](https://haematologica.org) | Calpain-1 docking and cleavage-induced pore remodelling |
| **2021** | [*Blood* 137(6):830-843](https://ashpublications.org/blood) | Cx62 hexamer + 62Gap27 mimetic peptide |
| **2021** | [*Proteins* 89:1607](https://onlinelibrary.wiley.com/doi/abs/10.1002/prot.26231) | CASP14 community paper — co-authored with David Baker, Demis Hassabis, John Jumper |
| **2024** | CASP16 top-methods | MultiFOLD3 server + ModFOLDdock2Q |
| **2014** | *Abstr Pap Am Chem Soc* 2014;247 | Zn²⁺-sensing peptide design (Yıldız Technical) |

**Full publication list:** 14 papers · H-index 9 · 530+ citations.

---

## Related teaching artefacts

| Artefact | What it is |
|---|---|
| **Kent_Combined_v2.pptx** | Full Stage-2 interview deck — 20 teaching slides + 13-slide lecture |
| **Kent_Speaker_Notes.md** | Slide-by-slide script with 20-minute timing and 12-question Q&A pre-mortem |
| **Kent_Combined_v2.md** | Long-form narrative companion (teaching plan + 5 LOs + rubric) |
| **PDHub_Mutagenesis_BIOS6380.ipynb** | This notebook |
| **PDHub_Predict_Evaluate_BIOS6380.ipynb** | Companion notebook: ESMFold predict WT + mutant, evaluate with Cα RMSD / TM-score / lDDT / clash + AlphaMissense cross-check |

---

## Acknowledgements

- **Mark Wass** (Kent · BIOS6380 convenor · 3DLigandSite, Phyre2) — host module convenor.
- **Haïfa Ben Messaoud** (InstaDeep / BioNTech) — TCR-pMHC variant pipeline collaborator.
- **Liam McGuffin** (Reading · McGuffin Lab) — PhD and postdoc host; ModFOLDdock2Q co-author.
- **The CASP / CAPE / CAMEO community** — for setting the bar that this curriculum teaches students to clear.
- **Students** — Sahli, Genç, Edmunds, Alharbi — who co-authored, broke the pipelines first, and made everything better.

---

*This notebook is part of the BIOS6380 worked example for the Stage 2 interview at the University of Kent · Lecturer in AI in Biology / Bioinformatics (SNS-070-26) · 15 May 2026.*

**License:** MIT — fork it, teach with it, ship better variants.

**Questions?** Open an issue on the repo or email — I read everything within 24 hours.

🙏  *Thank you for spending time with this.*

## Master comparison — BRCA1 A1708E worked example (recap)

What the code cells above produce, gathered into one table for the end of the notebook:

| metric | WT | MT | Δ / verdict | interpretation |
|---|---|---|---|---|
| Residue identity | Ala (A) | Glu (E) | A → E | hydrophobic → negatively charged |
| Charge (e) | 0.0 | −1.0 | −1.0 | net charge change at the position |
| Hydropathy (Kyte–Doolittle) | +1.8 | −3.5 | −5.3 | negative = more hydrophilic; \|Δ\| > 4 is a large swing |
| Side-chain volume (Å³) | 88.6 | 138.4 | +49.8 | +50 Å³ swap can strain a packed pocket |
| pLDDT @ residue 1708 | 95.0 | 94.0 | −1.00 | structure-model self-confidence at the residue |
| Cα RMSD (BRCT1 pocket) | — | — | 0.06 Å | sub-Ångström = structure-model SILENT |
| AlphaMissense (variant) | — | — | 0.94 (≥ 0.564) | LIKELY PATHOGENIC |
| ConSurf (1–9) | — | — | 9 / 9 | residue fully conserved across vertebrates |
| ACMG/AMP call | — | — | LIKELY PATHOGENIC | PS3 + PM1 + PP3 (Richards 2015) |

The numbers come from a live ESMFold API run on the BRCT1 domain plus the cached evidence streams. The master-table code cell above recomputes them for whatever variant you set in section 0.